# Modern Data Engineering for AI Systems — Capstone (all 5 deliverables)
**SDAIA Academy | Trainer: Mohammed Albeladi | Single-notebook build for Google Colab**

This notebook implements the full rubric in one place:

| # | Deliverable | Pts | Where |
|---|---|---|---|
| 1 | Ingestion (Kafka + schema validation) | 20 | `pipeline_lib.ingest_stage()` |
| 2 | Delta Lakehouse (Bronze/Silver/Gold, real MERGE) | 25 | `bronze_stage / silver_stage / gold_stage` |
| 3 | RAG pipeline (chunking, embeddings, hybrid search, reranking) | 25 | `rag_index_stage / rag_answer_query` |
| 4 | Orchestration (Airflow DAG) | 15 | `capstone_pipeline.py` DAG |
| 5 | Quality Gate + Lineage (Great Expectations + OpenLineage) | 15 | `quality_gate_stage` + `emit_lineage` used in every stage |

**Design choice:** every pipeline stage lives once, in `pipeline_lib.py`. The notebook cells below call those functions directly so you can see each deliverable work step by step — and the exact same functions are imported by the Airflow DAG in the Orchestration section, so there's no logic duplicated between "the demo" and "the real pipeline."

**Colab caveats (read once):**
- Colab gives you a single ephemeral Linux VM with no persistent background services. Cells below download and launch **real** Kafka (KRaft mode, no Zookeeper needed) and **real** Airflow (`airflow standalone`) as background processes for the life of this session — genuine libraries, not simulations, just self-hosted instead of `docker run`.
- If your Colab runtime disconnects/restarts, re-run from the "Install everything" cell down — nothing persists across a fresh runtime.
- **Great Expectations' and OpenLineage's Python APIs change fairly often between versions.** The code below targets recent (2026) releases and includes a fallback for OpenLineage's transport. If an exact method name doesn't match what pip installs for you, paste me the error and I'll adjust the two or three lines that need it.
- Keep the **executed** notebook (outputs intact) — that's your evidence per the rubric's "run your code and keep the output" requirement. Don't clear outputs before committing to GitHub.


## 1. Install everything

In [ ]:
%%capture
!pip install -q kafka-python pydantic
!pip install -q pyspark==3.5.1 delta-spark==3.1.0
!pip install -q chromadb sentence-transformers rank_bm25
!pip install -q "great-expectations>=1.0,<2.0"
!pip install -q openlineage-python
!pip install -q anthropic
!apt-get install -y -qq openjdk-17-jre-headless > /dev/null

import sys
PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
AIRFLOW_VERSION = "2.9.3"
CONSTRAINT_URL = f"https://raw.githubusercontent.com/apache/airflow/constraints-{AIRFLOW_VERSION}/constraints-{PY_VER}.txt"
!pip install -q "apache-airflow=={AIRFLOW_VERSION}" --constraint "{CONSTRAINT_URL}"


In [ ]:
print("All installs finished.")

## 2. `pipeline_lib.py` — the shared pipeline module

One file, imported both by the notebook (below) and by the Airflow DAG (Section 6). Contains:
- OpenLineage helper (`emit_lineage`, `lineage_stage` decorator — wraps every stage with START/COMPLETE/FAIL events)
- The Pydantic data contract (Deliverable 1)
- `ingest_stage` (Deliverable 1)
- `bronze_stage`, `silver_stage`, `gold_stage` (Deliverable 2)
- `quality_gate_stage` (Deliverable 5)
- `rag_index_stage`, `hybrid_search`, `rerank`, `rag_answer_query` (Deliverable 3)


In [ ]:
%%writefile pipeline_lib.py
"""
pipeline_lib.py — shared stage functions for the capstone pipeline.
Used directly from the notebook for step-by-step demonstration, and imported
by the Airflow DAG (capstone_pipeline.py) so both paths run identical logic.
"""
import json
import os
import re
import time
from datetime import datetime, timezone
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

# ---------------------------------------------------------------------------
# OpenLineage helper (Deliverable 5 — lineage)
# ---------------------------------------------------------------------------
from openlineage.client import OpenLineageClient
from openlineage.client.run import RunEvent, RunState, Run, Job
from openlineage.client.uuid import generate_new_uuid


def _build_ol_client():
    """Real OpenLineageClient. Prefers the built-in ConsoleTransport; falls back
    to a minimal print-transport if that class moved in your installed version."""
    try:
        from openlineage.client.transport.console import ConsoleTransport
        return OpenLineageClient(transport=ConsoleTransport())
    except Exception:
        from openlineage.client.transport.base import Transport

        class _PrintTransport(Transport):
            kind = "print"

            def emit(self, event):
                try:
                    print("[OpenLineage]", event.to_json())
                except Exception:
                    print("[OpenLineage]", event)

        return OpenLineageClient(transport=_PrintTransport())


_ol_client = _build_ol_client()
NAMESPACE = "capstone-pipeline"


def emit_lineage(job_name: str, state, run_id: str = None):
    run_id = run_id or str(generate_new_uuid())
    event = RunEvent(
        eventType=state,
        eventTime=datetime.now(timezone.utc).isoformat(),
        run=Run(runId=run_id),
        job=Job(namespace=NAMESPACE, name=job_name),
        producer="https://github.com/OpenLineage/OpenLineage/tree/main/client/python",
    )
    try:
        _ol_client.emit(event)
    except Exception as e:
        print(f"[OpenLineage] emit failed for {job_name}/{state}: {e}")
    return run_id


def lineage_stage(job_name):
    """Decorator: emits START before, COMPLETE after success, FAIL on exception."""
    def decorator(fn):
        def wrapper(*args, **kwargs):
            run_id = emit_lineage(job_name, RunState.START)
            try:
                result = fn(*args, **kwargs)
            except Exception:
                emit_lineage(job_name, RunState.FAIL, run_id)
                raise
            emit_lineage(job_name, RunState.COMPLETE, run_id)
            return result
        return wrapper
    return decorator


# ---------------------------------------------------------------------------
# Demo flags (used to prove the quality gate actually blocks bad data)
# ---------------------------------------------------------------------------
def _get_flag(name, default="false"):
    try:
        from airflow.models import Variable
        return Variable.get(name, default_var=default).lower() == "true"
    except Exception:
        path = f"/content/{name}.flag"
        return os.path.exists(path) and open(path).read().strip().lower() == "true"


def set_flag(name, value: bool):
    """Notebook-side helper to toggle demo flags before Airflow is running."""
    with open(f"/content/{name}.flag", "w") as f:
        f.write("true" if value else "false")


# ---------------------------------------------------------------------------
# Deliverable 1: Ingestion — data contract + Kafka producer/consumer
# ---------------------------------------------------------------------------
class OrderEvent(BaseModel):
    """Data contract enforced at the ingestion boundary."""
    order_id: str = Field(min_length=1)
    customer_id: str = Field(min_length=1)
    amount: float = Field(gt=0)
    currency: Literal["SAR", "USD", "EUR"]
    status: Literal["created", "paid", "shipped", "cancelled"]
    event_time: datetime


RAW_RECORDS = [
    {"order_id": "ord-1001", "customer_id": "cus-01", "amount": 249.99, "currency": "SAR", "status": "paid", "event_time": "2026-09-10T09:00:00"},
    {"order_id": "ord-1002", "customer_id": "cus-02", "amount": 75.50, "currency": "USD", "status": "created", "event_time": "2026-09-10T09:01:00"},
    {"order_id": "ord-1003", "amount": 40.0, "currency": "SAR", "status": "paid", "event_time": "2026-09-10T09:02:00"},          # missing customer_id
    {"order_id": "ord-1004", "customer_id": "cus-03", "amount": -10.0, "currency": "SAR", "status": "paid", "event_time": "2026-09-10T09:03:00"},  # negative amount
    {"order_id": "ord-1005", "customer_id": "cus-04", "amount": 20.0, "currency": "GBP", "status": "paid", "event_time": "2026-09-10T09:04:00"},   # invalid currency
    {"order_id": "ord-1006", "customer_id": "cus-05", "amount": 30.0, "currency": "SAR", "status": "paid", "event_time": "not-a-date"},            # bad timestamp
]

KAFKA_BOOTSTRAP = "localhost:9092"


@lineage_stage("ingestion")
def ingest_stage(records=None):
    from kafka import KafkaProducer, KafkaConsumer

    records = records if records is not None else RAW_RECORDS

    producer = KafkaProducer(bootstrap_servers=KAFKA_BOOTSTRAP,
                              value_serializer=lambda v: json.dumps(v).encode())
    for rec in records:
        producer.send("events.raw", value=rec)
    producer.flush()

    consumer = KafkaConsumer(
        "events.raw", bootstrap_servers=KAFKA_BOOTSTRAP,
        auto_offset_reset="earliest", group_id=f"validator-{int(time.time()*1000)}",
        value_deserializer=lambda v: json.loads(v.decode()),
        consumer_timeout_ms=8000,
    )
    router = KafkaProducer(bootstrap_servers=KAFKA_BOOTSTRAP,
                            value_serializer=lambda v: json.dumps(v).encode())

    valid, rejected = [], []
    for msg in consumer:
        payload = msg.value
        try:
            validated = OrderEvent(**payload)
            router.send("events.valid", value=validated.model_dump(mode="json"))
            valid.append(validated.model_dump(mode="json"))
        except ValidationError as e:
            reason = e.errors()
            router.send("events.deadletter", value={
                "original_payload": payload, "rejection_reason": reason,
                "quarantined_at": datetime.now(timezone.utc).isoformat(),
            })
            rejected.append({"payload": payload, "reason": reason})
    router.flush()

    print(f"[ingestion] valid={len(valid)} rejected={len(rejected)}")
    for r in rejected:
        print(f"  [REJECTED] {r['payload'].get('order_id')}: {r['reason'][0]['msg']}")
    return {"valid": valid, "rejected": rejected}


# ---------------------------------------------------------------------------
# Spark + Delta session
# ---------------------------------------------------------------------------
_spark = None


def get_spark():
    global _spark
    if _spark is not None:
        return _spark
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip

    builder = (
        SparkSession.builder.appName("capstone-lakehouse")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.driver.memory", "4g")
    )
    _spark = configure_spark_with_delta_pip(builder).getOrCreate()
    _spark.sparkContext.setLogLevel("ERROR")
    return _spark


LAKE_ROOT = "/content/lakehouse"
BRONZE_PATH = f"{LAKE_ROOT}/bronze_orders"
SILVER_PATH = f"{LAKE_ROOT}/silver_orders"
GOLD_PATH = f"{LAKE_ROOT}/gold_order_summary"


# ---------------------------------------------------------------------------
# Deliverable 2: Bronze — schema-enforced append
# ---------------------------------------------------------------------------
@lineage_stage("bronze_layer")
def bronze_stage(valid_records):
    from pyspark.sql import Row

    spark = get_spark()
    now = datetime.now(timezone.utc).isoformat()
    rows = [Row(order_id=r["order_id"], customer_id=r["customer_id"], amount=float(r["amount"]),
                currency=r["currency"], status=r["status"], event_time=r["event_time"], ingested_at=now)
            for r in valid_records]

    if _get_flag("inject_bad_batch"):
        rows.append(Row(order_id="ord-POISON", customer_id="cus-99", amount=-5.0,
                         currency="SAR", status="paid", event_time="2026-09-10T10:00:00", ingested_at=now))
        print("[bronze] WARNING: inject_bad_batch=true -> appended a poisoned row on purpose (for the quality-gate demo)")

    df = spark.createDataFrame(rows).selectExpr(
        "order_id", "customer_id", "amount", "currency", "status",
        "CAST(event_time AS TIMESTAMP) as event_time", "CAST(ingested_at AS TIMESTAMP) as ingested_at",
    )
    df.write.format("delta").mode("append").option("mergeSchema", "false").save(BRONZE_PATH)
    total = spark.read.format("delta").load(BRONZE_PATH).count()
    print(f"[bronze] appended {df.count()} rows — bronze table now has {total} rows total")
    return total


# ---------------------------------------------------------------------------
# Deliverable 5: Quality gate (Great Expectations) — gates bronze before promotion
# ---------------------------------------------------------------------------
@lineage_stage("quality_gate")
def quality_gate_stage():
    import great_expectations as gx

    spark = get_spark()
    bronze_pdf = spark.read.format("delta").load(BRONZE_PATH).toPandas()

    context = gx.get_context(mode="ephemeral")
    data_source = context.data_sources.add_pandas("bronze_pandas")
    data_asset = data_source.add_dataframe_asset(name="bronze_df")
    batch_def = data_asset.add_batch_definition_whole_dataframe("bronze_batch")
    batch = batch_def.get_batch(batch_parameters={"dataframe": bronze_pdf})

    suite = gx.ExpectationSuite(name="bronze_quality_suite")
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id"))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(column="currency", value_set=["SAR", "USD", "EUR"]))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="amount", min_value=0, strict_min=True))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
        column="status", value_set=["created", "paid", "shipped", "cancelled"]))

    result = batch.validate(suite)
    if not result.success:
        failed = [r["expectation_config"]["kwargs"] for r in result.results if not r["success"]]
        raise RuntimeError(f"Quality gate FAILED — {len(failed)} expectation(s) violated: {failed}")

    print(f"[quality_gate] PASSED — {len(result.results)} expectations checked against {len(bronze_pdf)} bronze rows")
    return True


# ---------------------------------------------------------------------------
# Deliverable 2: Silver — real MERGE (upsert) keyed on order_id
# ---------------------------------------------------------------------------
@lineage_stage("silver_layer")
def silver_stage():
    from delta.tables import DeltaTable
    from pyspark.sql import Window, functions as F

    spark = get_spark()
    bronze_df = spark.read.format("delta").load(BRONZE_PATH).filter("order_id != 'ord-POISON'")

    w = Window.partitionBy("order_id").orderBy(F.col("ingested_at").desc())
    dedup_df = (bronze_df.withColumn("rn", F.row_number().over(w))
                          .filter("rn = 1").drop("rn"))

    if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
        dedup_df.write.format("delta").mode("overwrite").save(SILVER_PATH)
        print(f"[silver] created silver table with {dedup_df.count()} rows")
    else:
        silver_table = DeltaTable.forPath(spark, SILVER_PATH)
        (silver_table.alias("s")
            .merge(dedup_df.alias("b"), "s.order_id = b.order_id")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        metrics = silver_table.history(1).select("operationMetrics").collect()[0]["operationMetrics"]
        print(f"[silver] MERGE complete — {metrics}")

    total = spark.read.format("delta").load(SILVER_PATH).count()
    print(f"[silver] silver table now has {total} rows")
    return total


# ---------------------------------------------------------------------------
# Deliverable 2: Gold — genuine aggregate (not a copy of Silver)
# ---------------------------------------------------------------------------
@lineage_stage("gold_layer")
def gold_stage():
    from pyspark.sql import functions as F

    spark = get_spark()
    silver_df = spark.read.format("delta").load(SILVER_PATH)
    gold_df = (silver_df.groupBy("currency", "status")
               .agg(F.sum("amount").alias("total_amount"),
                    F.count("*").alias("order_count"),
                    F.avg("amount").alias("avg_order_value"))
               .orderBy("currency", "status"))
    gold_df.write.format("delta").mode("overwrite").save(GOLD_PATH)
    print("[gold] wrote aggregate table:")
    gold_df.show(truncate=False)
    return gold_df.count()


# ---------------------------------------------------------------------------
# Deliverable 3: RAG pipeline
# ---------------------------------------------------------------------------
DOCS = {
    "doc_returns": "Our return policy allows customers to return unused items within 30 days of delivery for a full refund. "
                   "Items must be in original packaging. Refunds are issued to the original payment method within 5-7 business days.",
    "doc_shipping": "Standard shipping within Saudi Arabia takes 2-4 business days. Express shipping delivers next-day in Riyadh, "
                    "Jeddah, and Dammam. International orders take 7-14 business days depending on customs clearance.",
    "doc_payments": "We accept Mada, Visa, Mastercard, Apple Pay, and STC Pay. Payments are processed securely and orders are only "
                    "marked as paid once payment confirmation is received from the payment gateway.",
    "doc_loyalty": "The loyalty program awards 1 point per 10 SAR spent. Points can be redeemed at checkout for discounts starting "
                   "at 100 points. Points expire 12 months after they are earned if unused.",
    "doc_cancellation": "Orders can be cancelled free of charge before they are marked as shipped. Once an order has shipped, it "
                        "can no longer be cancelled and must go through the standard return process instead.",
    "doc_support": "Customer support is available via live chat and phone from 9am to 9pm daily. Response time for email "
                   "inquiries is typically under 24 hours on business days.",
}

CHROMA_DIR = "/content/chroma_db"

_bm25 = None
_bm25_chunks = None
_embedder = None
_reranker = None


def _chunk_text(doc_id, text, chunk_size=1):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    chunks = []
    for i in range(0, len(sentences), chunk_size):
        window = sentences[i:i + chunk_size]
        if not window:
            continue
        chunks.append({"chunk_id": f"{doc_id}_{i}", "doc_id": doc_id, "text": " ".join(window)})
    return chunks


@lineage_stage("rag_indexing")
def rag_index_stage():
    global _bm25, _bm25_chunks, _embedder
    import chromadb
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi

    all_chunks = []
    for doc_id, text in DOCS.items():
        all_chunks.extend(_chunk_text(doc_id, text))

    _embedder = _embedder or SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = _embedder.encode([c["text"] for c in all_chunks]).tolist()

    client = chromadb.PersistentClient(path=CHROMA_DIR)
    try:
        client.delete_collection("docs")
    except Exception:
        pass
    collection = client.create_collection("docs")
    collection.add(
        ids=[c["chunk_id"] for c in all_chunks],
        embeddings=embeddings,
        documents=[c["text"] for c in all_chunks],
        metadatas=[{"doc_id": c["doc_id"]} for c in all_chunks],
    )

    _bm25_chunks = all_chunks
    _bm25 = BM25Okapi([c["text"].lower().split() for c in all_chunks])

    print(f"[rag_indexing] indexed {len(all_chunks)} chunks into Chroma (dense) + BM25 (keyword)")
    return len(all_chunks)


def _reciprocal_rank_fusion(rank_lists, k=60):
    scores = {}
    for ranked_ids in rank_lists:
        for rank, cid in enumerate(ranked_ids):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def hybrid_search(query, top_k_each=5, fused_top_n=6):
    import chromadb

    global _embedder
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    collection = client.get_collection("docs")

    if _embedder is None:
        from sentence_transformers import SentenceTransformer
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
    q_emb = _embedder.encode([query]).tolist()
    dense_res = collection.query(query_embeddings=q_emb, n_results=top_k_each)
    dense_ids = dense_res["ids"][0]

    bm25_scores = _bm25.get_scores(query.lower().split())
    bm25_ranked = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:top_k_each]
    bm25_ids = [_bm25_chunks[i]["chunk_id"] for i in bm25_ranked]

    fused = _reciprocal_rank_fusion([dense_ids, bm25_ids])[:fused_top_n]
    id_to_chunk = {c["chunk_id"]: c for c in _bm25_chunks}
    return [id_to_chunk[cid] for cid, _score in fused]


def rerank(query, candidates, top_n=3):
    global _reranker
    from sentence_transformers import CrossEncoder
    _reranker = _reranker or CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    pairs = [(query, c["text"]) for c in candidates]
    scores = _reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [c for c, _s in ranked[:top_n]]


@lineage_stage("rag_answer")
def rag_answer_query(query):
    candidates = hybrid_search(query)
    top_chunks = rerank(query, candidates, top_n=3)
    context_block = "\n".join(f"[{c['chunk_id']}] {c['text']}" for c in top_chunks)

    answer = None
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if api_key:
        try:
            import anthropic
            client = anthropic.Anthropic(api_key=api_key)
            resp = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=400,
                messages=[{"role": "user", "content":
                    f"Answer the question using ONLY the context below. Cite chunk ids like [doc_x_0] inline.\n\n"
                    f"Context:\n{context_block}\n\nQuestion: {query}"}],
            )
            answer = "".join(b.text for b in resp.content if b.type == "text")
        except Exception as e:
            print(f"[rag_answer] LLM call failed, falling back to extractive answer: {e}")

    if not answer:
        answer = ("Grounded context (no ANTHROPIC_API_KEY set — showing retrieved passages with citations):\n" +
                  "\n".join(f"- {c['text']} [{c['chunk_id']}]" for c in top_chunks))

    print(f"[rag_answer] Q: {query}\n{answer}\n")
    return {"query": query, "answer": answer, "citations": [c["chunk_id"] for c in top_chunks]}


## 3. Start a real Kafka broker (KRaft mode, no Zookeeper)

In [ ]:
import pathlib, subprocess, time

KAFKA_VERSION = "3.7.1"
SCALA_VERSION = "2.13"
KAFKA_DIR = f"/content/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"

if not pathlib.Path(KAFKA_DIR).exists():
    !wget -q https://downloads.apache.org/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz
    !tar -xzf kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz -C /content

cluster_id = subprocess.run([f"{KAFKA_DIR}/bin/kafka-storage.sh", "random-uuid"],
                             capture_output=True, text=True).stdout.strip()
!{KAFKA_DIR}/bin/kafka-storage.sh format -t {cluster_id} -c {KAFKA_DIR}/config/kraft/server.properties --ignore-formatted

kafka_log = open("/content/kafka.log", "w")
kafka_proc = subprocess.Popen([f"{KAFKA_DIR}/bin/kafka-server-start.sh", f"{KAFKA_DIR}/config/kraft/server.properties"],
                               stdout=kafka_log, stderr=subprocess.STDOUT)
print(f"Kafka starting (pid={kafka_proc.pid})...")
time.sleep(20)
!tail -n 10 /content/kafka.log


In [ ]:
TOPICS = ["events.raw", "events.valid", "events.deadletter"]
for t in TOPICS:
    !{KAFKA_DIR}/bin/kafka-topics.sh --create --if-not-exists --topic {t} \
        --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
!{KAFKA_DIR}/bin/kafka-topics.sh --list --bootstrap-server localhost:9092


## 4. Deliverable 1 & 2 — Ingestion, Bronze, Quality Gate, Silver (MERGE), Gold

Run the cells in order. The second block deliberately sends an **update** for `ord-1001` (status `paid` → `shipped`) to prove the Silver MERGE is a real upsert, not just an append. The third block deliberately turns on `inject_bad_batch` to prove the quality gate actually **blocks** bad data from reaching Silver/Gold.

In [ ]:
import pipeline_lib as lib
import importlib
importlib.reload(lib)

# --- First pass: ingest the baseline batch, land it in bronze, gate it, promote to silver/gold ---
ingest_result = lib.ingest_stage()
lib.bronze_stage(ingest_result["valid"])
lib.quality_gate_stage()
lib.silver_stage()
lib.gold_stage()


In [ ]:
# --- Prove the MERGE is a real upsert: re-send ord-1001 with an updated status ---
updated_batch = [{**lib.RAW_RECORDS[0], "status": "shipped"}]  # ord-1001: paid -> shipped
validated_updated = [lib.OrderEvent(**updated_batch[0]).model_dump(mode="json")]

lib.bronze_stage(validated_updated)
lib.quality_gate_stage()
lib.silver_stage()   # should UPDATE ord-1001 in place, not duplicate it
lib.gold_stage()     # aggregate reflects the status change


In [ ]:
# --- Prove the quality gate actually gates: inject a poisoned row and watch it get blocked ---
lib.set_flag("inject_bad_batch", True)
lib.bronze_stage([])  # empty valid batch, but the poison row still gets appended by the flag

try:
    lib.quality_gate_stage()
    print("Unexpected: gate passed (it should have failed).")
except RuntimeError as e:
    print("Quality gate correctly BLOCKED the pipeline:")
    print(e)
    print("\n(silver_stage/gold_stage are NOT called — this is exactly what the Airflow DAG in Section 6 enforces automatically.)")
finally:
    lib.set_flag("inject_bad_batch", False)


## 5. Deliverable 3 — RAG pipeline (chunking, embeddings, hybrid search + RRF, reranking, grounded answers)

In [ ]:
lib.rag_index_stage()

In [ ]:
import os
# Optional: paste an Anthropic API key to get a real generated, cited answer.
# Leave blank and press Enter to skip -- you'll get an extractive fallback answer instead (still grounded + cited).
from getpass import getpass
key = getpass("Anthropic API key (optional, press Enter to skip): ")
if key.strip():
    os.environ["ANTHROPIC_API_KEY"] = key.strip()


In [ ]:
lib.rag_answer_query("What is the return policy for unused items?")
lib.rag_answer_query("How long does international shipping take?")
lib.rag_answer_query("Can I cancel my order after it has shipped?")


## 6. Deliverable 4 — Orchestration (Airflow DAG wiring every stage together)

`ingest >> bronze >> quality_gate >> silver >> gold >> rag_index`, with the default Airflow trigger rule (`all_success`) meaning a failed `quality_gate` leaves `silver`, `gold`, and `rag_index` in `upstream_failed` — they never run. We'll prove that with a failing run, then a clean run.

In [ ]:
import os, subprocess, time

os.environ["AIRFLOW_HOME"] = "/content/airflow"
AIRFLOW_HOME = os.environ["AIRFLOW_HOME"]
os.makedirs(f"{AIRFLOW_HOME}/dags", exist_ok=True)

# pipeline_lib.py must be importable from the dags folder too
!cp /content/pipeline_lib.py {AIRFLOW_HOME}/dags/pipeline_lib.py

airflow_log = open("/content/airflow_standalone.log", "w")
airflow_proc = subprocess.Popen(["airflow", "standalone"], stdout=airflow_log, stderr=subprocess.STDOUT,
                                 env={**os.environ})
print(f"Airflow standalone starting (pid={airflow_proc.pid}) -- this takes 1-2 minutes on first run (db init + webserver + scheduler)...")
time.sleep(90)
!grep -i "standalone | Login with username" /content/airflow_standalone.log || tail -n 20 /content/airflow_standalone.log


In [ ]:
%%writefile {AIRFLOW_HOME}/dags/capstone_pipeline.py
"""
capstone_pipeline.py — Airflow DAG wiring every capstone stage together.
Deliverable 4 (Orchestration): correct task dependencies so a failed quality
gate halts the pipeline before downstream stages run (Deliverable 5).
"""
from datetime import datetime

from airflow import DAG
from airflow.operators.python import PythonOperator

import pipeline_lib as lib

default_args = {"owner": "capstone", "retries": 0}

with DAG(
    dag_id="capstone_pipeline",
    default_args=default_args,
    schedule=None,
    start_date=datetime(2026, 1, 1),
    catchup=False,
    tags=["capstone", "modern-data-engineering"],
) as dag:

    def _ingest(**context):
        result = lib.ingest_stage()
        context["ti"].xcom_push(key="valid_records", value=result["valid"])

    def _bronze(**context):
        valid_records = context["ti"].xcom_pull(key="valid_records", task_ids="ingest")
        lib.bronze_stage(valid_records)

    def _quality_gate(**context):
        lib.quality_gate_stage()

    def _silver(**context):
        lib.silver_stage()

    def _gold(**context):
        lib.gold_stage()

    def _rag_index(**context):
        lib.rag_index_stage()

    t_ingest = PythonOperator(task_id="ingest", python_callable=_ingest)
    t_bronze = PythonOperator(task_id="bronze", python_callable=_bronze)
    t_quality_gate = PythonOperator(task_id="quality_gate", python_callable=_quality_gate)
    t_silver = PythonOperator(task_id="silver", python_callable=_silver)
    t_gold = PythonOperator(task_id="gold", python_callable=_gold)
    t_rag_index = PythonOperator(task_id="rag_index", python_callable=_rag_index)

    t_ingest >> t_bronze >> t_quality_gate >> t_silver >> t_gold >> t_rag_index


In [ ]:
# Give the scheduler a moment to parse the new DAG file
import time
time.sleep(30)
!airflow dags list | grep capstone_pipeline


### 6a. Run 1 — with `inject_bad_batch=true`: the gate should fail and halt downstream

In [ ]:
!airflow variables set inject_bad_batch true
!airflow dags trigger capstone_pipeline
import time
time.sleep(60)
!airflow dags list-runs -d capstone_pipeline | head -n 5


In [ ]:
RUN_ID = !airflow dags list-runs -d capstone_pipeline -o plain | tail -n 1 | awk '{print $2}'
run_id = RUN_ID[0]
print("run_id:", run_id)
!airflow tasks states-for-dag-run capstone_pipeline {run_id}


### 6b. Run 2 — with `inject_bad_batch=false`: full pipeline should succeed end to end

In [ ]:
!airflow variables set inject_bad_batch false
!airflow dags trigger capstone_pipeline
import time
time.sleep(90)
RUN_ID2 = !airflow dags list-runs -d capstone_pipeline -o plain | tail -n 1 | awk '{print $2}'
run_id2 = RUN_ID2[0]
print("run_id:", run_id2)
!airflow tasks states-for-dag-run capstone_pipeline {run_id2}


You should see Run 1 show `quality_gate` as **failed** and `silver` / `gold` / `rag_index` as **upstream_failed** (never executed), and Run 2 show all six tasks as **success**. That side-by-side is your evidence for Deliverable 4 (correct dependencies) and Deliverable 5 (the gate actually gates).

Airflow's web UI is running on port 8080 inside the Colab VM if you want to look at the graph view — forward it with `from google.colab.output import eval_js; print(eval_js("google.colab.kernel.proxyPort(8080)"))` and open the printed URL. The admin login/password were printed near the top of the Airflow standalone log in the cell above.

## 7. Rubric checklist

- **Ingestion (20 pts):** real Kafka broker + `kafka-python`, Pydantic data contract, malformed records quarantined to `events.deadletter` with the exact rejection reason — proven in Section 4 and again for free every time `ingest_stage` runs inside the DAG.
- **Delta Lakehouse (25 pts):** Bronze/Silver/Gold on real `delta-spark`, a genuine `MERGE ... whenMatchedUpdateAll ... whenNotMatchedInsertAll` keyed on `order_id` (proven by the `ord-1001` status-update cell), Gold is a `groupBy` aggregate, not a copy of Silver.
- **RAG pipeline (25 pts):** sentence-level chunking, `sentence-transformers` embeddings in a real Chroma vector store, `rank_bm25` keyword search, Reciprocal Rank Fusion, `CrossEncoder` reranking, and a grounded, cited answer (LLM-generated if you supplied a key, extractive-with-citations otherwise).
- **Orchestration (15 pts):** a real Airflow DAG with `ingest >> bronze >> quality_gate >> silver >> gold >> rag_index`, triggered and shown in both a failing and a succeeding run.
- **Quality Gate + Lineage (15 pts):** Great Expectations suite that raises and actually halts the DAG when violated; OpenLineage `START`/`COMPLETE`/`FAIL` events emitted around every single stage via the `lineage_stage` decorator.

**Before you submit:** keep this notebook's outputs intact, push it plus `pipeline_lib.py` and the DAG file to GitHub with a proper README (setup, how to run, architecture overview), incremental commits, a `.gitignore`, and the training-program attribution (SDAIA Academy, program name, cohort dates) that section 2.2 of the rubric requires. I can draft that README for you next if you want it.
